# fact_price_tick

Live price ticks from CoinGecko via the Lakeflow pipeline. Run after the pipeline `team-crypto-ticks-streaming` has updated `ticks_silver`.

Grain: **one asset x one moment**. Source: `ticks_silver`.

Quality gate before writing: row count = distinct business keys in silver, every dimension key found,
unique primary key, every date inside `dim_date`. Any problem stops the notebook, nothing is written.

`volume_24h` is a rolling 24h USD volume: do not sum it across ticks and do not compare it with `fact_ohlc_hourly.volume` (coins per hour).

In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F
from utilities.gold_transformations import FACT_TICK_COLUMNS, build_fact_price_tick, fact_quality_issues

spark.conf.set("spark.sql.session.timeZone", "UTC")

silver_df = spark.table(SILVER_TABLE)
dim_asset = spark.table(DIM_ASSET)
dim_source = spark.table(DIM_SOURCE)
dim_date = spark.table(DIM_DATE)

fact_df = build_fact_price_tick(silver_df, dim_asset, dim_source)

### Quality gate

In [0]:
expected_rows = silver_df.select("symbol", "event_time").distinct().count()

issues = fact_quality_issues(
    fact_df,
    expected_rows=expected_rows,
    key_columns=["asset_key", "event_time"],
    dim_columns=["asset_key", "date_key", "source_key"],
    calendar=dim_date,
)
if issues:
    raise ValueError("fact_price_tick quality gate failed: " + "; ".join(issues))

### Write (idempotent full overwrite)

In [0]:
(fact_df
    .select(*FACT_TICK_COLUMNS)
    .withColumn("processed_at", F.current_timestamp())
    .createOrReplaceTempView("fact_src"))

cols = ", ".join(FACT_TICK_COLUMNS + ["processed_at"])
spark.sql(f"INSERT OVERWRITE {GOLD}.fact_price_tick ({cols}) SELECT {cols} FROM fact_src")